# 06 - Preparación del dataset para Power BI

## Objetivo

En este notebook se preparará la versión autorizada del dataset que será utilizada como fuente para el dashboard en Power BI.

La preparación incluye:

- cargar el dataset limpio generado en etapas anteriores;
- aplicar la protección del identificador del estudiante;
- eliminar el identificador original `student_id`;
- conservar un identificador seudonimizado para análisis;
- validar la estructura final del dataset;
- guardar una versión publicable para Power BI.

El objetivo es asegurar que el dashboard utilice una fuente protegida y coherente con las decisiones de gobernanza definidas previamente.

In [73]:
import pandas as pd
import hashlib
from pathlib import Path

In [74]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_PROCESSED = DATA_DIR / "procesados"

ruta_dataset_limpio = (
    DATA_PROCESSED / "edtech_dataset_limpio.csv"
)

ruta_dataset_publicable = (
    DATA_PROCESSED / "edtech_dataset_publicable.csv"
)

## 1. Carga y validación inicial del dataset

Se carga el dataset limpio generado en las etapas anteriores para verificar que contiene la información necesaria antes de crear la versión publicable para Power BI.

En esta revisión se valida:

- que el archivo pueda cargarse correctamente;
- la cantidad de filas y columnas;
- la presencia del identificador original `student_id`;
- una muestra inicial de los datos.

Esta comprobación permite asegurar que la protección del identificador se aplique sobre la fuente correcta.

In [75]:
df = pd.read_csv(ruta_dataset_limpio)

print("Dataset cargado correctamente")
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Dataset cargado correctamente
Número de filas: 200000
Número de columnas: 36


In [76]:
print(
    "student_id presente:",
    "student_id" in df.columns
)

student_id presente: True


In [77]:
df.head()

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Video,1,1,0,1,2024,10,43,6.53,0.7837
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Mixed,1,1,1,1,2024,11,46,28.05,0.8064
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Text-based,1,1,0,0,2024,6,26,20.25,0.8422
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Mixed,0,1,0,1,2024,11,47,4.57,0.7217
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Video,1,1,1,1,2024,6,26,13.68,0.8467


## 2. Protección del identificador del estudiante

Para evitar exponer directamente el identificador original del estudiante en Power BI, se genera un identificador seudonimizado utilizando una función hash.

El nuevo campo `student_id_hash` permitirá mantener la relación entre los registros de un mismo estudiante sin mostrar `student_id`.

Posteriormente, el identificador original será eliminado de la versión publicable.

In [78]:
df_publicable = df.copy()

In [79]:
df_publicable["student_id_hash"] = (
    df_publicable["student_id"]
    .astype(str)
    .apply(
        lambda valor: hashlib.sha256(
            valor.encode()
        ).hexdigest()[:12]
    )
)

In [80]:
df_publicable[
    ["student_id", "student_id_hash"]
].head()

,student_id,student_id_hash
0,student_02020,b260928134ea
1,student_07410,f9c061ec9a42
2,student_03086,6528a9f2b19e
3,student_05092,f08582d4a22b
4,student_03626,ffe2fb94ce9b


In [81]:
df_publicable = df_publicable.drop(
    columns=["student_id"]
)

In [82]:
print(
    "student_id presente:",
    "student_id" in df_publicable.columns
)

print(
    "student_id_hash presente:",
    "student_id_hash" in df_publicable.columns
)

student_id presente: False
student_id_hash presente: True


## 3. Selección de campos para la versión publicable

Antes de guardar el dataset que será utilizado en Power BI, se revisan las columnas disponibles para conservar únicamente aquellas que aportan valor al análisis.

El identificador original `student_id` ya fue eliminado y sustituido por `student_id_hash`.

Para esta versión se mantienen las variables académicas, de interacción, curso y desempeño necesarias para construir KPIs y visualizaciones. Se evita eliminar campos que serán utilizados posteriormente para segmentación, comparación o análisis temporal.

In [83]:
print("Columnas disponibles en la versión publicable:\n")

for columna in df_publicable.columns:
    print(columna)

Columnas disponibles en la versión publicable:

interaction_id
course_id
timestamp
activity_type
duration_seconds
completion_rate
score
attempts
device_used
help_sought
peer_interaction
age
education_level
study_field
learning_style
device_preference
timezone
previous_online_courses
has_learning_disability
motivation_level
course_title
subject_area
difficulty_level
duration_weeks
instructor_rating
course_format
has_assignments
has_quizzes
has_discussion_forum
certificate_offered
interaction_year
interaction_month
interaction_week
duration_minutes
average_student_score
student_id_hash


In [84]:
columnas_publicables = [
    "interaction_id",
    "student_id_hash",

    "age",
    "education_level",
    "study_field",
    "learning_style",
    "device_preference",
    "timezone",
    "previous_online_courses",
    "has_learning_disability",
    "motivation_level",

    "course_id",
    "course_title",
    "subject_area",
    "difficulty_level",
    "duration_weeks",
    "instructor_rating",
    "course_format",
    "has_assignments",
    "has_quizzes",
    "has_discussion_forum",
    "certificate_offered",

    "timestamp",
    "activity_type",
    "duration_seconds",
    "completion_rate",
    "score",
    "attempts",
    "device_used",
    "help_sought",
    "peer_interaction",

    "interaction_year",
    "interaction_month",
    "interaction_week",
    "duration_minutes",
    "average_student_score"
]

In [85]:
columnas_faltantes = [
    columna
    for columna in columnas_publicables
    if columna not in df_publicable.columns
]

print("Columnas faltantes:", columnas_faltantes)

Columnas faltantes: []


In [86]:
df_publicable = df_publicable[
    columnas_publicables
].copy()

In [87]:
print("Número de columnas finales:", df_publicable.shape[1])

df_publicable.head()

Número de columnas finales: 36


,interaction_id,student_id_hash,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,...,score,attempts,device_used,help_sought,peer_interaction,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score
0,interaction_000000,b260928134ea,24,Undergraduate,Other,Reading/Writing,Desktop,UTC,5,0,...,NaN,1,Tablet,False,False,2024,10,43,6.53,0.7837
1,interaction_000001,f9c061ec9a42,33,Professional,Arts,Auditory,Mobile,UTC-5,7,1,...,0.9,2,Mobile,False,False,2024,11,46,28.05,0.8064
2,interaction_000002,6528a9f2b19e,27,Graduate,Other,Auditory,Mobile,UTC+1,7,0,...,NaN,1,Desktop,False,False,2024,6,26,20.25,0.8422
3,interaction_000003,f08582d4a22b,44,Graduate,Other,Visual,Mobile,UTC-5,5,0,...,NaN,1,Desktop,False,False,2024,11,47,4.57,0.7217
4,interaction_000004,ffe2fb94ce9b,20,Graduate,Business,Kinesthetic,Desktop,UTC-8,6,1,...,NaN,1,Mobile,False,False,2024,6,26,13.68,0.8467


## 4. Validación final y guardado del dataset publicable

Antes de utilizar el dataset en Power BI, se realiza una validación final para comprobar que:

- el identificador original `student_id` ya no esté presente;
- el identificador seudonimizado `student_id_hash` sí esté disponible;
- no existan columnas esperadas faltantes;
- el dataset conserve registros para análisis;
- el archivo pueda guardarse correctamente como fuente autorizada para visualización.

La versión final será almacenada como:

`datos/procesados/edtech_dataset_publicable.csv`

In [88]:
print("Filas finales:", df_publicable.shape[0])
print("Columnas finales:", df_publicable.shape[1])

print(
    "student_id presente:",
    "student_id" in df_publicable.columns
)

print(
    "student_id_hash presente:",
    "student_id_hash" in df_publicable.columns
)

print(
    "Duplicados completos:",
    df_publicable.duplicated().sum()
)

print(
    "Total de valores nulos:",
    df_publicable.isnull().sum().sum()
)

Filas finales: 200000
Columnas finales: 36
student_id presente: False
student_id_hash presente: True
Duplicados completos: 0
Total de valores nulos: 133457


In [89]:
df_publicable.to_csv(
    ruta_dataset_publicable,
    index=False
)

print("Dataset publicable guardado correctamente:")
print(ruta_dataset_publicable)

Dataset publicable guardado correctamente:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_dataset_publicable.csv


In [90]:
df_validacion = pd.read_csv(
    ruta_dataset_publicable
)

print("Archivo validado correctamente")
print("Filas:", df_validacion.shape[0])
print("Columnas:", df_validacion.shape[1])

print(
    "student_id presente:",
    "student_id" in df_validacion.columns
)

print(
    "student_id_hash presente:",
    "student_id_hash" in df_validacion.columns
)

Archivo validado correctamente
Filas: 200000
Columnas: 36
student_id presente: False
student_id_hash presente: True


## 5. Preguntas de negocio

El dashboard de EdTech Learning Analytics estará orientado a responder preguntas relacionadas con desempeño, participación y comportamiento académico.

Las cinco preguntas principales son:

1. ¿Cuál es el promedio general de calificaciones de los estudiantes?
2. ¿Qué cursos presentan mayor y menor rendimiento académico promedio?
3. ¿Cómo evoluciona el nivel de participación de los estudiantes a lo largo del tiempo?
4. ¿Cuál es el nivel promedio de finalización de las actividades académicas?
5. ¿Qué proporción de estudiantes presenta un desempeño inferior a su promedio individual o señales de menor participación?

## 6. KPIs propuestos

### KPI 1. Promedio general de calificaciones

**Definición:** promedio de las calificaciones registradas en actividades evaluables.

**Campo principal:** `score`

**Unidad:** porcentaje o valor decimal.

**Pregunta que responde:** ¿Cuál es el rendimiento académico promedio general?

---

### KPI 2. Tasa promedio de finalización

**Definición:** promedio del porcentaje de finalización de las actividades registradas.

**Campo principal:** `completion_rate`

**Unidad:** porcentaje.

**Pregunta que responde:** ¿Qué nivel de avance presentan los estudiantes en las actividades?

---

### KPI 3. Total de interacciones

**Definición:** cantidad total de interacciones registradas en la plataforma educativa.

**Campo principal:** `interaction_id`

**Unidad:** número de interacciones.

**Pregunta que responde:** ¿Cuál es el nivel general de actividad dentro de la plataforma?

---

### KPI 4. Estudiantes activos

**Definición:** cantidad de estudiantes distintos que presentan al menos una interacción registrada.

**Campo principal:** `student_id_hash`

**Unidad:** número de estudiantes.

**Pregunta que responde:** ¿Cuántos estudiantes participan en el periodo seleccionado?

---

### KPI 5. Duración promedio de interacción

**Definición:** tiempo promedio dedicado a cada interacción dentro de la plataforma.

**Campo principal:** `duration_minutes`

**Unidad:** minutos.

**Pregunta que responde:** ¿Cuánto tiempo dedican en promedio los estudiantes a cada interacción?

---

## 7. KPI complementario para análisis académico

Como indicador adicional puede utilizarse:

### Actividades evaluadas

**Definición:** número de actividades que contienen una calificación válida.

**Campo principal:** `score`

**Unidad:** número de actividades.

**Pregunta que responde:** ¿Cuántas actividades evaluables existen en el contexto seleccionado?

## 8. Validación del dashboard

Se realizaron comprobaciones para verificar que los indicadores y visualizaciones de Power BI fueran consistentes con el dataset de origen.

Las validaciones incluyeron:

- comparación del promedio general de calificaciones con el campo `score`;
- validación del total de interacciones;
- validación del número de estudiantes distintos mediante `student_id_hash`;
- prueba de los segmentadores por periodo, curso y nivel de dificultad;
- revisión de filtros aplicados a nivel de visual, página y reporte;
- confirmación de que el identificador original `student_id` no esté presente en el modelo.

Estas comprobaciones permiten asegurar que el dashboard refleje correctamente los datos preparados y que la información mostrada responda de forma coherente a los filtros seleccionados.

## 9. Hallazgos del dashboard

A partir de las visualizaciones construidas en Power BI se identificaron tres hallazgos principales. Cada uno se documenta considerando la evidencia observada, el contexto de análisis, su interpretación, una posible decisión y sus limitaciones.

### Hallazgo 1. Rendimiento académico por curso

**Evidencia:**  
El curso **[nombre del curso]** presenta el promedio de calificación más alto dentro del periodo analizado, con un valor aproximado de **[valor]**.

**Contexto:**  
Análisis realizado considerando los filtros activos del dashboard y el promedio de `score` por curso.

**Interpretación:**  
Este resultado indica que, dentro del contexto seleccionado, los estudiantes obtienen un rendimiento académico promedio superior en este curso en comparación con los demás cursos analizados.

**Decisión posible:**  
Revisar las características del curso, su formato, nivel de dificultad y diseño de actividades para identificar prácticas que puedan ser útiles en otros cursos.

**Limitación:**  
Un promedio alto no permite concluir por sí solo que el diseño del curso sea la causa del mejor rendimiento, ya que pueden intervenir otros factores relacionados con los estudiantes o las actividades evaluadas.

---

### Hallazgo 2. Evolución temporal del rendimiento

**Evidencia:**  
Durante **[mes o periodo]** se observa **[un aumento / una disminución / estabilidad]** en el promedio general de calificaciones, alcanzando aproximadamente **[valor]**.

**Contexto:**  
El resultado se obtiene a partir del gráfico de tendencia por `AñoMes`.

**Interpretación:**  
La variación muestra que el desempeño académico no permanece completamente constante a lo largo del periodo analizado.

**Decisión posible:**  
Revisar los periodos con variaciones más marcadas para identificar cambios en actividades, carga académica o comportamiento de participación.

**Limitación:**  
La tendencia muestra una asociación temporal, pero no permite establecer que el periodo sea la causa directa del cambio en las calificaciones.

---

### Hallazgo 3. Participación por tipo de actividad

**Evidencia:**  
La actividad **[tipo de actividad]** concentra el mayor número de interacciones, con aproximadamente **[cantidad]** registros.

**Contexto:**  
El resultado se obtiene mediante la comparación del total de interacciones por `activity_type`.

**Interpretación:**  
Este tipo de actividad representa una parte importante de la participación registrada dentro de la plataforma educativa.

**Decisión posible:**  
Considerar este tipo de actividad como un punto relevante para futuros análisis de participación y comportamiento académico.

**Limitación:**  
Un mayor número de interacciones no implica necesariamente un mejor rendimiento académico ni una mayor calidad del aprendizaje.

## 10. Checklist final de la etapa

| Requerimiento | Estado |
|---|---|
| Dataset publicable y protegido utilizado como fuente | Cumplido |
| Modelo semántico construido | Cumplido |
| Tabla de hechos `FactInteracciones` | Cumplido |
| Dimensión de fechas `DimFecha` | Cumplido |
| Dimensión de cursos `DimCurso` | Cumplido |
| Dimensión de estudiantes `DimEstudiante` | Cumplido |
| Relaciones 1:* configuradas y activas | Cumplido |
| Al menos cinco KPIs definidos | Cumplido |
| Medidas DAX creadas | Cumplido |
| Página de Resumen Ejecutivo | Cumplido |
| Página de Análisis Académico | Cumplido |
| Página de Detalle | Cumplido |
| Gráfico de tendencia temporal | Cumplido |
| Comparación de rendimiento por curso | Cumplido |
| Análisis por tipo de actividad | Cumplido |
| Matriz curso-estudiante agregada | Cumplido |
| Al menos tres segmentadores | Cumplido |
| Validación de indicadores y filtros | Cumplido |
| Identificador original `student_id` excluido | Cumplido |
| Tres hallazgos documentados | Cumplido |
| Limitaciones de los hallazgos documentadas | Cumplido |

## 11. Conclusión

En esta etapa se construyó la capa de visualización del proyecto EdTech Learning Analytics utilizando Power BI.

Como punto de partida se generó una versión publicable del dataset en la que el identificador original del estudiante fue eliminado y sustituido por un identificador seudonimizado, manteniendo así la continuidad con las decisiones de gobernanza definidas previamente.

En Power BI se construyó un modelo semántico compuesto por una tabla principal de interacciones y dimensiones relacionadas con fechas, cursos y estudiantes. Este modelo permitió organizar los datos y facilitar la aplicación de filtros y cálculos dinámicos.

Se definieron medidas DAX para representar indicadores relacionados con:

- promedio de calificaciones;
- tasa promedio de finalización;
- total de interacciones;
- estudiantes activos;
- duración promedio de interacción;
- actividades evaluadas.

El dashboard fue organizado en tres niveles de análisis:

- **Resumen Ejecutivo**, orientado a mostrar los principales indicadores;
- **Análisis Académico**, enfocado en tendencias, rendimiento por curso y participación;
- **Detalle**, orientado a explorar información más específica mediante matrices y comparaciones.

También se incorporaron segmentadores por periodo, curso y nivel de dificultad, permitiendo que los indicadores y visualizaciones respondan dinámicamente al contexto seleccionado.

Finalmente, se realizaron validaciones para comprobar la consistencia de los resultados con el dataset de origen y se documentaron tres hallazgos considerando evidencia, contexto, interpretación, posibles decisiones y limitaciones.

Con esta etapa, los datos preparados durante las fases anteriores fueron transformados en una herramienta visual que facilita la exploración del rendimiento y la participación académica, manteniendo los criterios de privacidad y gobernanza definidos para el proyecto.